# DPO Training for Collaboration Facilitator Assistant

In [ ]:
USE_GOOGLE_COLAB = False

In [ ]:
if USE_GOOGLE_COLAB:
    from google.colab import drive
    import sys
    drive.mount('/content/drive', force_remount=True)
    sys.path.append('/content/drive/MyDrive/cs329x/')

    from google.colab import userdata
    huggingface_token = userdata.get("HUGGINGFACE_TOKEN")

    data_directory = '/content/drive/MyDrive/cs329x'

    !pip install -U trl
    !pip install -U bitsandbytes

else:
    import os
    huggingface_token = os.environ['HUGGINGFACE_TOKEN']

    data_directory = './results'

In [ ]:
import json
from src import utils
from datasets import Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

## Load data

In [ ]:
# Load documents
documents = utils.load_dataset_from_jsonl(f'{data_directory}/training_docs.jsonl')
print(f"Loaded {len(documents)} documents")

In [ ]:
# Load the generated training data and build HuggingFace datasets

training_data = []
with open(f'{data_directory}/training_data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        training_data.append(json.loads(line.strip()))

print(f"Loaded {len(training_data)} training examples")
print(f"\nFirst example keys: {list(training_data[0].keys())}")

# Convert to DPO format
dpo_data = []
for item in training_data:
    dpo_data.append({
        'prompt': item['prompt'],
        'chosen': item['accepted'],
        'rejected': item['rejected']
    })

# Create HuggingFace dataset
dataset = Dataset.from_list(dpo_data)
print(f"Created dataset with {len(dataset)} examples")
print(f"\nExample:")
print(dataset[0])

## Train

In [ ]:
# Model configuration
output_dir = f"{data_directory}/dpo_qwen_model"
model_name = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading model: {model_name}")
print(f"Output directory: {output_dir}")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

In [ ]:
# Load model with 4-bit quantization for efficiency
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True,
)

print("Model loaded with 4-bit quantization")
print(f"Model device: {model.device}")

In [ ]:
# Prepare model for training with LoRA
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoRA adapters configured and applied")

In [ ]:
# Create a reference model (frozen copy for DPO)
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True,
)

print("Reference model loaded")

In [ ]:
# Configure DPO training
training_args = DPOConfig(
    output_dir=output_dir,
    num_train_epochs=1, # Changed from 3 to 1 (when adding 10x data)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32, # Changed from 4 to 32
    learning_rate=1e-6, # changed from 5e-5 to 1e-6 to avoid overfitting
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    remove_unused_columns=False,
    beta=0.25,  # DPO temperature parameter - change from 0.1 to 0.25 to avoid overfitting
    max_prompt_length=2048,
    max_length=2560,
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  DPO beta: {training_args.beta}")

In [ ]:
# Initialize DPO trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("DPO Trainer initialized")

In [ ]:
# Train the model
print("Starting DPO training...")
dpo_trainer.train()
print("\nTraining complete!")

## Save

In [ ]:
final_model_path = f"{data_directory}/dpo_qwen_final"

# Save the fine-tuned model
dpo_trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Save the fine-tuned model
dpo_trainer.save_model(f'{data_directory}/eval_generations_all_type_combos_model/')
tokenizer.save_pretrained(f'{data_directory}/eval_generations_all_type_combos_model/')

print(f"Model saved to {final_model_path}")